In [ ]:
#Please note that the following code was run in Kaggle's environment

In [ ]:
%%writefile dataset.py

import os
import torch
import cv2
import numpy as np
from torch.utils.data import Dataset
import torchvision.transforms.functional as F

class DubaiDataset(Dataset): 
    def __init__(self, img_dir, mask_dir): #the paths to the images and masks folders 
        self.img_dir = img_dir 
        self.mask_dir = mask_dir
        self.imgs = list(sorted(os.listdir(img_dir))) #Sort to make sure images and masks line up perfectly
        self.masks = list(sorted(os.listdir(mask_dir)))

    def __len__(self): #Returns the number of samples in the dataset
        return len(self.imgs)

    def __getitem__(self, idx): #Returns the image and target for a given index
        img_path = os.path.join(self.img_dir, self.imgs[idx]) 
        mask_path = os.path.join(self.mask_dir, self.masks[idx])
        
        img = cv2.imread(img_path) # Load image
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) #Convert from BGR to RGB
        
x        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) # Load binary mask
        
        num_instances, instance_mask = cv2.connectedComponents(mask) # Separate the connected blobs into individual instances
        obj_ids = np.unique(instance_mask)[1:] # Drop background (0)
        
        masks = instance_mask == obj_ids[:, None, None] # Create a binary mask for each instance
        
        num_objs = len(obj_ids) # Get bounding boxes for each instance
        boxes = []
        for i in range(num_objs):
            pos = np.where(masks[i]) # Get the pixel locations of the mask
            xmin = np.min(pos[1])
            xmax = np.max(pos[1])
            ymin = np.min(pos[0])
            ymax = np.max(pos[0])
            
            # Catch 0-pixel width and height edge cases
            if xmax == xmin: xmax += 1
            if ymax == ymin: ymax += 1
                
            boxes.append([xmin, ymin, xmax, ymax])

        # Convert to PyTorch Tensors
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.ones((num_objs,), dtype=torch.int64) 
        masks = torch.as_tensor(masks, dtype=torch.uint8)
        image_id = torch.tensor([idx])
        
        # Calculate area as required by Mask R-CNN
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) #width * height

        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks,
            "image_id": image_id,
            "area": area
        }
        
        img_tensor = F.to_tensor(img)

        return img_tensor, target

Writing dataset.py


In [ ]:
%%writefile model.py

import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def get_baseline_model(num_classes=2):
    # Load a model pre-trained on COCO
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

    in_features = model.roi_heads.box_predictor.cls_score.in_features #Get the number of input features for the classifier
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes) #Replace the pre-trained head with a new one

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels #Get the number of input features for the mask classifier
    hidden_layer = 256 #Choose a number of hidden layers for the mask classifier
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    return model

Writing model.py


In [ ]:
%%writefile train.py

import torch
from torch.utils.data import DataLoader
from dataset import DubaiDataset 
from model import get_baseline_model


def collate_fn(batch): #Custom collate function to handle batches of varying sizes since each image can have a different number of buildings
    return tuple(zip(*batch))

def train_model():
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu') #Use GPU if available, otherwise fall back to CPU
    print(f"Training on: {device}") 

    dataset = DubaiDataset(img_dir='data/train/images', mask_dir='data/train/masks')
    
    data_loader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=2, collate_fn=collate_fn)

    model = get_baseline_model(num_classes=2) #Get the model with the modified heads 
    model.to(device) #Move the model to the device being used (CPU/GPU)

    params = [p for p in model.parameters() if p.requires_grad] #Only optimize the parameters that require gradients
    optimizer = torch.optim.Adam(params, lr=0.0001)

    num_epochs = 5
    for epoch in range(num_epochs):
        model.train() 
        epoch_loss = 0 #Track the total loss for the epoch
        
        for images, targets in data_loader:
            images = list(image.to(device) for image in images) 
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets] 

            loss_dict = model(images, targets) #Returns a dictionary of losses for the different components (box regression, classification, mask loss)
            losses = sum(loss for loss in loss_dict.values()) #Combine the different losses into a single value for backpropagation
            epoch_loss += losses.item()

            optimizer.zero_grad() #Zero the gradients before backpropagation
            losses.backward() #Backpropagate the loss to compute gradients
            optimizer.step() #Update the model parameters based on the computed gradients

        print(f"Epoch: {epoch+1}/{num_epochs}, Average Loss: {epoch_loss/len(data_loader):.4f}")

    torch.save(model.state_dict(), 'baseline_maskrcnn.pth')
    print("Training complete. Baseline weights saved to baseline_maskrcnn.pth.")

if __name__ == '__main__':
    train_model()

Overwriting train.py


In [ ]:
import cv2
import os
import numpy as np
import shutil

def patch_dubai_dataset_kaggle(base_dir, out_img_dir, out_mask_dir, patch_size=256):
    # Nuke the old folders completely to prevent leftover ghost images
    if os.path.exists(out_img_dir): shutil.rmtree(out_img_dir)
    if os.path.exists(out_mask_dir): shutil.rmtree(out_mask_dir)
    
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_mask_dir, exist_ok=True)
    
    # The exact dark purple building color we confirmed worked on your local test
    target_color = np.array([152, 16, 60]) 
    lower_bound = np.clip(target_color - 15, 0, 255)
    upper_bound = np.clip(target_color + 15, 0, 255)
    
    patch_count = 0
    
    # Use os.walk to recursively hunt down the dataset, bypassing Kaggle's folder naming
    for root, dirs, files in os.walk(base_dir):
        # Look for the 'images' subdirectories
        if os.path.basename(root) == 'images':
            mask_dir = root.replace('images', 'masks')
            
            if not os.path.exists(mask_dir):
                continue
                
            print(f"Found and scanning folder: {root}...")
            
            for img_name in files:
                # Ensure we only process valid image files
                if img_name.endswith('.jpg') or img_name.endswith('.png'):
                    img_path = os.path.join(root, img_name)
                    
                    # The MBRSC masks use .png even if the raw images are .jpg
                    mask_name = img_name.replace('.jpg', '.png')
                    mask_path = os.path.join(mask_dir, mask_name)
                    
                    if not os.path.exists(mask_path):
                        continue
                        
                    img = cv2.imread(img_path)
                    mask = cv2.imread(mask_path)
                    
                    if img is None or mask is None:
                        continue
                        
                    h, w = img.shape[:2]
                    
                    # Slice the images into patches
                    for y in range(0, h, patch_size):
                        for x in range(0, w, patch_size):
                            img_patch = img[y:y+patch_size, x:x+patch_size]
                            mask_patch = mask[y:y+patch_size, x:x+patch_size]
                            
                            if img_patch.shape[:2] == (patch_size, patch_size):
                                binary_mask = cv2.inRange(mask_patch, lower_bound, upper_bound)
                                
                                if cv2.countNonZero(binary_mask) > 100: 
                                    patch_filename = f"patch_{patch_count}.png"
                                    cv2.imwrite(os.path.join(out_img_dir, patch_filename), img_patch)
                                    cv2.imwrite(os.path.join(out_mask_dir, patch_filename), binary_mask)
                                    patch_count += 1

    print(f"Success! Generated {patch_count} training patches.")

# Point it to the root Kaggle input directory and let the script find the dataset naturally
base_directory = '/kaggle/input/'

patch_dubai_dataset_kaggle(
    base_dir=base_directory, 
    out_img_dir='/kaggle/working/data/train/images', 
    out_mask_dir='/kaggle/working/data/train/masks'
)

Found and scanning folder: /kaggle/input/datasets/humansintheloop/semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 7/images...
Found and scanning folder: /kaggle/input/datasets/humansintheloop/semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 8/images...
Found and scanning folder: /kaggle/input/datasets/humansintheloop/semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 2/images...
Found and scanning folder: /kaggle/input/datasets/humansintheloop/semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 5/images...
Found and scanning folder: /kaggle/input/datasets/humansintheloop/semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 1/images...
Found and scanning folder: /kaggle/input/datasets/humansintheloop/semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 3/images...
Found and scanning folder: /kaggle/input/datasets/humansintheloop/semantic-segment

In [ ]:
!python train.py

Training on: cuda
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth
100%|█████████████████████████████████████████| 170M/170M [00:00<00:00, 196MB/s]
Epoch: 1/5, Average Loss: 1.2783
Epoch: 2/5, Average Loss: 1.0540
Epoch: 3/5, Average Loss: 0.9538
Epoch: 4/5, Average Loss: 0.8489
Epoch: 5/5, Average Loss: 0.7573
Training complete. Baseline weights saved to baseline_maskrcnn.pth.


In [ ]:
from IPython.display import FileLink

FileLink(r'baseline_maskrcnn.pth')

/kaggle/working/baseline_maskrcnn.pth

In [ ]:
%%writefile fsanet.py

import torch
import torch.nn as nn
import torch.fft

class FrequencySelfAttention(nn.Module): #The FsaNet module that will be injected into the ResNet backbone
    def __init__(self, in_channels, k=16): #number of channels in the input feature map, k is the size of the low-frequency patch to attend to
        super().__init__()
        self.k = k 
        self.in_channels = in_channels
        
        self.query_conv = nn.Conv2d(in_channels, in_channels // 8, kernel_size=1)
        self.key_conv = nn.Conv2d(in_channels, in_channels // 8, kernel_size=1)
        self.value_conv = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.shape
        

        x_freq = torch.fft.rfft2(x, norm='ortho')

        k_h = min(self.k, x_freq.shape[2])
        k_w = min(self.k, x_freq.shape[3])
        low_freq = x_freq[:, :, :k_h, :k_w]
        
        low_freq_real = torch.view_as_real(low_freq).mean(dim=-1)
        
        proj_query = self.query_conv(low_freq_real).view(B, -1, k_h * k_w).permute(0, 2, 1)
        proj_key = self.key_conv(low_freq_real).view(B, -1, k_h * k_w)
        proj_value = self.value_conv(low_freq_real).view(B, -1, k_h * k_w)
        
        energy = torch.bmm(proj_query, proj_key)
        attention = torch.softmax(energy, dim=-1)
        out = torch.bmm(proj_value, attention.permute(0, 2, 1))
        
        out = out.view(B, C, k_h, k_w)
        out_complex = torch.complex(out, torch.zeros_like(out))
        
        padded_freq = torch.zeros_like(x_freq)
        padded_freq[:, :, :k_h, :k_w] = out_complex
        
        out_spatial = torch.fft.irfft2(padded_freq, s=(H, W), norm='ortho')
        
        return x + self.gamma * out_spatial

Writing fsanet.py


In [ ]:
%%writefile newmodel.py

import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from fsanet import FrequencySelfAttention
import torch

def get_fsanet_model(num_classes=2):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

    # ResNet50's final layer output has 2048 channels.
    # We wrap the existing layer and our new FsaNet module into a Sequence.
    backbone_final_layer = model.backbone.body.layer4
    
    # Replace layer4 with itself plus the frequency attention module
    model.backbone.body.layer4 = torch.nn.Sequential(
        backbone_final_layer,
        FrequencySelfAttention(in_channels=2048, k=16) 
    )

    # Replace the box predictor head (same as baseline)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Replace the mask predictor head (same as baseline)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    return model

Writing newmodel.py


In [ ]:
%%writefile newtrain.py

import torch
from torch.utils.data import DataLoader
from dataset import DubaiDataset 
from newmodel import get_fsanet_model

def collate_fn(batch):
    return tuple(zip(*batch))

def train_model():
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    print(f"Training on: {device}")

    dataset = DubaiDataset(img_dir='data/train/images', mask_dir='data/train/masks')
    
    data_loader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=2, collate_fn=collate_fn)

    model = get_fsanet_model(num_classes=2)
    model.to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=0.0001)

    num_epochs = 5
    for epoch in range(num_epochs):
        model.train() 
        epoch_loss = 0
        
        for images, targets in data_loader:
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            epoch_loss += losses.item()

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

        print(f"Epoch: {epoch+1}/{num_epochs}, Average Loss: {epoch_loss/len(data_loader):.4f}")

        torch.save(model.state_dict(), 'fsanet_maskrcnn.pth')    
        print("Training complete. Baseline weights saved to baseline_maskrcnn.pth.")

if __name__ == '__main__':
    train_model()

Overwriting newtrain.py


In [ ]:
!python newtrain.py

Training on: cuda
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth
100%|█████████████████████████████████████████| 170M/170M [00:01<00:00, 176MB/s]
Epoch: 1/5, Average Loss: 1.2646
Training complete. Baseline weights saved to baseline_maskrcnn.pth.
Epoch: 2/5, Average Loss: 1.0450
Training complete. Baseline weights saved to baseline_maskrcnn.pth.
Epoch: 3/5, Average Loss: 0.9329
Training complete. Baseline weights saved to baseline_maskrcnn.pth.
Epoch: 4/5, Average Loss: 0.8610
Training complete. Baseline weights saved to baseline_maskrcnn.pth.
Epoch: 5/5, Average Loss: 0.7666
Training complete. Baseline weights saved to baseline_maskrcnn.pth.
